# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [16]:
# Clone your specific repository
!git clone https://github.com/PrathamDudani/FlyRank_Assignment.git

# Change into the repository's folder
%cd FlyRank_Assignment

Cloning into 'FlyRank_Assignment'...
remote: Enumerating objects: 147, done.
remote: Counting objects: 100% (147/147), done.
remote: Compressing objects: 100% (117/117), done.
remote: Total 147 (delta 49), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (147/147), 1.95 MiB | 16.13 MiB/s, done.
Resolving deltas: 100% (49/49), done.
/content/FlyRank_Assignment/FlyRank_Assignment


In [17]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
valid = df[df["avg_position"] > 0].copy()

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Given your lane (CTR/engagement opportunity scoring) and that your baseline already outputs a binary action (REVIEW_FOR_REFRESH / NO_ACTION), the natural fit is a classification model predicting the same kind of outcome — most likely is_declining_label, since that's the dataset's actual outcome label (and it's meant to be predicted, not used as a feature).

Start with Logistic Regression as your primary model — it's interpretable, coefficients map cleanly to reason-code-style explanations, and it's the fair comparison point against a hand-written rule. Optionally add Decision Tree or Random Forest with permutation importance as a stretch if Logistic Regression underperforms — but don't reach for Random Forest just because it scores higher; the skill explicitly says "does not reward complexity alone."

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Use a grouped split by client_id, not a random row split. Reason: your data has 32 clients with many rows each — a random split would leak the same client's other content items into both train and test, making the model look better than it'd generalize to a genuinely new client. This is the same principle as the leakage warning in your data skill.

In [18]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(valid, groups=valid["client_id"]))
train, test = valid.iloc[train_idx], valid.iloc[test_idx]

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [19]:
print(df["trend_direction"].value_counts(dropna=False))

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


In [20]:
valid["is_declining_label"] = (valid["trend_direction"] == "declining").astype(int)

In [22]:
print(df["trend_direction"].unique())
print(valid["trend_direction"].unique())
print(valid["trend_direction"].value_counts())

['down' 'stable' 'new' 'up' 'flat']
['down' 'stable' 'up' 'new' 'flat']
trend_direction
down      16254
stable     5962
up         4388
flat       1109
new        1082
Name: count, dtype: int64


In [23]:
valid["is_declining_label"] = (valid["trend_direction"] == "down").astype(int)
print(valid["is_declining_label"].value_counts())

is_declining_label
1    16254
0    12541
Name: count, dtype: int64


In [24]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, roc_auc_score

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(valid, groups=valid["client_id"]))
train, test = valid.iloc[train_idx], valid.iloc[test_idx]

print(train["is_declining_label"].value_counts())
print(test["is_declining_label"].value_counts())

features = ["ctr", "avg_position", "impressions_90d", "engagement_rate",
            "days_since_last_update", "search_volume"]

X_train, y_train = train[features].fillna(0), train["is_declining_label"]
X_test, y_test = test[features].fillna(0), test["is_declining_label"]

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train, y_train)
pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:,1]

print("Model precision:", precision_score(y_test, pred))
print("Model recall:", recall_score(y_test, pred))
print("Model AUC:", roc_auc_score(y_test, proba))
print("Base rate (test):", y_test.mean())

is_declining_label
1    13111
0     9863
Name: count, dtype: int64
is_declining_label
1    3143
0    2678
Name: count, dtype: int64
Model precision: 0.5756244853143014
Model recall: 0.6671969455933822
Model AUC: 0.5422848336821136
Base rate (test): 0.5399415907919601


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [25]:
from sklearn.inspection import permutation_importance

r = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42)
for i in r.importances_mean.argsort()[::-1]:
    print(f"{features[i]}: {r.importances_mean[i]:.4f}")

# look at false positives / false negatives
errors = test.copy()
errors["pred"] = pred
errors["true"] = y_test.values
fp = errors[(errors["pred"]==1)&(errors["true"]==0)]
fn = errors[(errors["pred"]==0)&(errors["true"]==1)]
print("False positives:", len(fp), "| False negatives:", len(fn))
print(fp[features].describe())

avg_position: 0.0327
impressions_90d: 0.0234
engagement_rate: 0.0060
ctr: 0.0041
search_volume: 0.0008
days_since_last_update: 0.0003
False positives: 1546 | False negatives: 1046
               ctr  avg_position  impressions_90d  engagement_rate  \
count  1546.000000   1546.000000      1546.000000      1546.000000   
mean      0.170977      8.231048      2053.659120         1.547581   
std       0.344968      6.106097      4999.143923         4.394523   
min       0.000000      0.300000         1.000000         0.000000   
25%       0.000000      4.900000         4.000000         0.000000   
50%       0.000000      7.000000        58.500000         0.000000   
75%       0.220000      9.700000      1826.250000         0.000000   
max       3.280000     67.000000     54464.000000        50.000000   

       days_since_last_update  search_volume  
count             1546.000000    1391.000000  
mean                39.164295      83.968368  
std                 41.306685     716.348401  
m

Test

In [26]:
from sklearn.model_selection import train_test_split

# Naive random row split — same features, same model, no grouping
train_naive, test_naive = train_test_split(
    valid, test_size=0.2, random_state=42, stratify=valid["is_declining_label"]
)

X_train_n, y_train_n = train_naive[features].fillna(0), train_naive["is_declining_label"]
X_test_n, y_test_n = test_naive[features].fillna(0), test_naive["is_declining_label"]

model_naive = LogisticRegression(max_iter=1000, class_weight="balanced")
model_naive.fit(X_train_n, y_train_n)
pred_n = model_naive.predict(X_test_n)
proba_n = model_naive.predict_proba(X_test_n)[:, 1]

print("=== Naive random split ===")
print("Precision:", precision_score(y_test_n, pred_n))
print("Recall:", recall_score(y_test_n, pred_n))
print("AUC:", roc_auc_score(y_test_n, proba_n))

print("\n=== Grouped split (client_id) ===")
print("Precision:", precision_score(y_test, pred))
print("Recall:", recall_score(y_test, pred))
print("AUC:", roc_auc_score(y_test, proba))

# check overlap that the naive split allows and the grouped split doesn't
train_clients = set(train_naive["client_id"])
test_clients = set(test_naive["client_id"])
print("\nClients appearing in BOTH naive train and test:", len(train_clients & test_clients))

=== Naive random split ===
Precision: 0.6019307211811471
Recall: 0.6521070439864657
AUC: 0.5662654037991991

=== Grouped split (client_id) ===
Precision: 0.5756244853143014
Recall: 0.6671969455933822
AUC: 0.5422848336821136

Clients appearing in BOTH naive train and test: 30


In [27]:
# Check whether any feature column is a direct or near-direct function of the label
# Look for suspiciously strong single-feature separation
for f in features:
    print(f, "-- correlation with label:", valid[f].corr(valid["is_declining_label"]))

# If avg_position or ctr were used to COMPUTE trend_direction upstream,
# check the raw data dictionary / generation script for this.
# Practical check: does trend_direction correlate near-perfectly with
# a combination of current vs prior period values, if those exist in the raw csv?
print(df.columns.tolist())  # look for *_prior, *_delta, *_change columns

ctr -- correlation with label: -0.06880418953698919
avg_position -- correlation with label: -0.0813044269963262
impressions_90d -- correlation with label: -0.032947445960446874
engagement_rate -- correlation with label: -0.02233770178979403
days_since_last_update -- correlation with label: 0.053194656735573186
search_volume -- correlation with label: -0.019990748631303016
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier

In [28]:
# Don't silently zero-fill — it's informative that it's missing
valid["search_volume_missing"] = valid["search_volume"].isna().astype(int)
valid["search_volume"] = valid["search_volume"].fillna(valid["search_volume"].median())

features_v2 = features + ["search_volume_missing"]
# re-run train/test split and model with features_v2 instead of features

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.